# 5장 Release Decision 검토

## Goal

데이터, 모델, serving과 운영 증거를 하나의 release 판단으로 연결합니다. Candidate A는 배포하지 않고 Candidate B만 승인 overlay에 포함되는지 확인합니다. Candidate B model approval은 frozen evidence의 `APPROVE`로 유지하며, target API와 telemetry를 실제로 관측하지 못한 경우 operational deployment scope는 `target_pending`으로 기록합니다.

In [1]:
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").is_file(), "tta-aiqa repository root를 찾지 못했습니다."
{"repository_root": ROOT.name}


{'repository_root': 'tta-aiqa'}

In [2]:
import json
from pathlib import Path
import pandas as pd

canonical_path = ROOT / "docs/reference/evidence/model/revisions/v2/canonical-benchmark.json"
canonical = json.loads(canonical_path.read_text(encoding="utf-8"))
decisions = pd.DataFrame(canonical["decisions"]).set_index("profile")
decisions[["decision", "checks"]]

,decision,checks
profile,,
candidate-a,HOLD,"{'false_negative_reduction': False, 'pr_auc_vs..."
candidate-b,APPROVE,"{'false_negative_reduction': True, 'pr_auc_vs_..."


### 1. Post-test release manifest

The manifest is the publication authorization. It must bind the canonical decision,
final benchmark, pre-test freeze, approved model bundle, and MLflow run IDs without
rewriting the freeze.

In [3]:
import hashlib

release_manifest_path = ROOT / "docs/reference/evidence/model/revisions/v2/release-manifest.json"
final_benchmark_path = ROOT / "docs/reference/evidence/model/revisions/v2/final-benchmark.json"
release_manifest = json.loads(release_manifest_path.read_text(encoding="utf-8"))
sha256 = lambda path: hashlib.sha256(path.read_bytes()).hexdigest()

release_chain = {
    "release_status": release_manifest["release_status"],
    "approved_profile": release_manifest["approved_profile"],
    "canonical_digest_matches": release_manifest["canonical_evidence"]["sha256"]
    == sha256(canonical_path),
    "final_digest_matches": release_manifest["final_evidence"]["sha256"]
    == sha256(final_benchmark_path),
    "freeze_digest_matches": release_manifest["freeze_manifest"]["sha256"]
    == canonical["sealed_test"]["freeze_manifest_sha256"],
    "historical_reconciliation": release_manifest["historical_reconciliation"],
}
assert release_chain["release_status"] == "release_approved"
assert release_chain["approved_profile"] == "candidate-b"
assert all(
    release_chain[key]
    for key in ("canonical_digest_matches", "final_digest_matches", "freeze_digest_matches")
)
release_chain

{'release_status': 'release_approved',
 'approved_profile': 'candidate-b',
 'canonical_digest_matches': True,
 'final_digest_matches': True,
 'freeze_digest_matches': True,
 'historical_reconciliation': {'frozen_dvc_lock_sha256': '227309cf7a551d0725cbfcdb02b260dce86b2b5a4e98d2d73c7b02dbea11e749',
  'frozen_dvc_lock_snapshot_available': False,
  'original_release_freeze_sha256': 'e91099b2ec96b901776b8c540ded2a7cb1607f9e8c0840d290ae257064f43336',
  'reason': 'serialized_bundle_and_metadata_integrity_contract',
  'schema_version': 1,
  'verified_test_dataset_sha256': '051491069cd385789047031d87ad92f3ebf86868f7b2da410307b639b906aa77'}}

## Steps

### 1. 배포 overlay와 승인 결과 연결

In [4]:
import yaml

overlay_root = ROOT / "deploy/kubernetes/overlays"
overlays = {
    name: "\n".join(
        path.read_text(encoding="utf-8")
        for path in sorted((overlay_root / name).glob("*.yaml"))
    )
    for name in ("baseline", "candidate-b", "rollback")
}
base_identity = dict(
    line.split("=", maxsplit=1)
    for line in (ROOT / "deploy/kubernetes/base/config/model-identity.env")
    .read_text(encoding="utf-8")
    .splitlines()
    if line and not line.startswith("#")
)
candidate_kustomization = yaml.safe_load(
    (overlay_root / "candidate-b/kustomization.yaml").read_text(encoding="utf-8")
)
candidate_identity = next(
    item
    for item in candidate_kustomization["configMapGenerator"]
    if item["name"] == "model-identity"
)
candidate_digest = candidate_identity["literals"][0].split("=", maxsplit=1)[1]
expected_digests = {
    "baseline": base_identity["AIQA_KSERVE_EXPECTED_MODEL_SHA256"],
    "candidate-b": candidate_digest,
    "rollback": base_identity["AIQA_KSERVE_EXPECTED_MODEL_SHA256"],
}
profiles = {"baseline": "baseline", "candidate-b": "candidate-b", "rollback": "baseline"}
summary = pd.DataFrame([
    {
        "overlay": name,
        "contains_baseline": "baseline-f2576f12512a" in document,
        "contains_candidate_b": "candidate-b-c712a8e52344" in document,
        "contains_candidate_a": "candidate-a" in document,
        "expected_model_sha256": expected_digests[name],
        "manifest_model_sha256": release_manifest["model_bundles"][
            f"{profiles[name]}/model.joblib"
        ],
    }
    for name, document in overlays.items()
]).set_index("overlay")
summary


,contains_baseline,contains_candidate_b,contains_candidate_a,expected_model_sha256,manifest_model_sha256
overlay,,,,,
baseline,True,False,False,f2576f12512a490c9814e5238c3f0d2a421a21637a4b03...,f2576f12512a490c9814e5238c3f0d2a421a21637a4b03...
candidate-b,False,True,False,c712a8e52344c09a493629f8e6a0283c6c3139fb46cc53...,c712a8e52344c09a493629f8e6a0283c6c3139fb46cc53...
rollback,True,False,False,f2576f12512a490c9814e5238c3f0d2a421a21637a4b03...,f2576f12512a490c9814e5238c3f0d2a421a21637a4b03...


### 2. 판단 기록 구성

In [5]:
release_record = {
    "dataset_sha256": canonical["sealed_test"]["dataset_sha256"],
    "sealed_test_status": canonical["sealed_test"]["status"],
    "candidate_a": decisions.loc["candidate-a", "decision"],
    "candidate_b": decisions.loc["candidate-b", "decision"],
    "deployment_allowed": canonical["deployment_allowed"],
    "model_approval": {
        "candidate-a": decisions.loc["candidate-a", "decision"],
        "candidate-b": decisions.loc["candidate-b", "decision"],
    },
    "operational_deployment_scope": "target_pending",
    "current_recommendation": "target evidence collection",
    "false_approval_risk": "unobserved target bundle or telemetry condition",
    "false_hold_risk": "delaying a frozen-approved Candidate B without model evidence",
    "reassessment_on_identity_mismatch": "rollback_required",
    "approved_overlay": "candidate-b",
    "rollback_overlay": "rollback",
    "owner_next_action": "Serving/Platform collects target identity and telemetry.",
}
release_record

{'dataset_sha256': '051491069cd385789047031d87ad92f3ebf86868f7b2da410307b639b906aa77',
 'sealed_test_status': 'evaluated_once',
 'candidate_a': 'HOLD',
 'candidate_b': 'APPROVE',
 'deployment_allowed': True,
 'model_approval': {'candidate-a': 'HOLD', 'candidate-b': 'APPROVE'},
 'operational_deployment_scope': 'target_pending',
 'current_recommendation': 'target evidence collection',
 'reassessment_on_identity_mismatch': 'rollback_required',
 'approved_overlay': 'candidate-b',
 'rollback_overlay': 'rollback',
 'owner_next_action': 'Serving/Platform collects target identity and telemetry.'}

### 3. Live deployment identity

강사 승인 GitOps sync 뒤에만 `AIQA_RISK_API_URL`을 assigned URL로 설정합니다. Candidate B
rollout을 확인하는 경우에는 `AIQA_EXPECTED_PROFILE=candidate-b`도 명시합니다. API profile
mismatch는 rollback review를 열 수 있지만, profile 하나가 맞는다고 target verified라고 쓰지는
않습니다. 같은 identity의 telemetry time window가 있어야 operational conclusion을 확장할 수 있습니다.

In [6]:
import os

import requests

API_URL = os.getenv("AIQA_RISK_API_URL")
expected_profile = os.getenv("AIQA_EXPECTED_PROFILE")
if API_URL:
    try:
        response = requests.get(f"{API_URL.rstrip('/')}/v1/model", timeout=3)
        response.raise_for_status()
        deployed_identity = response.json()
        observed_profile = deployed_identity.get("profile")
        deployed_identity = {
            **deployed_identity,
            "expected_profile": expected_profile,
            "profile_matches": (
                expected_profile is None or observed_profile == expected_profile
            ),
        }
    except requests.RequestException as error:
        deployed_identity = {"status": "API_UNREACHABLE", "detail": str(error)}
else:
    deployed_identity = {
        "status": "URL_NOT_CONFIGURED",
        "next_action": "Use the instructor-provided Risk API URL after GitOps sync.",
    }
if deployed_identity.get("profile_matches") is False:
    release_record["operational_deployment_scope"] = "rollback_required"
    release_record["current_recommendation"] = "rollback review"
    release_record["operational_reason"] = "observed API profile differs from expected profile"
else:
    release_record["operational_reason"] = (
        "API identity alone is insufficient; target telemetry is still required."
    )
{
    "identity_observation": deployed_identity,
    "operational_deployment_scope": release_record["operational_deployment_scope"],
    "current_recommendation": release_record["current_recommendation"],
}

{'identity_observation': {'status': 'URL_NOT_CONFIGURED',
  'next_action': 'Use the instructor-provided Risk API URL after GitOps sync.'},
 'operational_deployment_scope': 'target_pending',
 'current_recommendation': 'target evidence collection'}

## Checks

In [7]:
assert release_record["candidate_a"] == "HOLD"
assert release_record["candidate_b"] == "APPROVE"
assert release_record["deployment_allowed"] is True
assert release_record["model_approval"]["candidate-b"] == "APPROVE"
assert release_record["operational_deployment_scope"] in {
    "target_pending",
    "rollback_required",
}
assert summary.loc["candidate-b", "contains_candidate_b"]
assert not summary["contains_candidate_a"].any()
assert summary.loc["rollback", "contains_baseline"]
assert (summary["expected_model_sha256"] == summary["manifest_model_sha256"]).all()
print("Release decision checks passed.")


Release decision checks passed.


## Next Steps

강사 승인 GitOps sequence에서 Candidate B overlay를 sync한 뒤 expected profile, immutable identity와 Grafana telemetry time window를 같은 release record에 capture합니다. profile mismatch, valid payload contract failure 또는 owner-validated live operational condition은 `rollback_required` review를 열 수 있습니다. target context나 Grafana credential이 없으면 rollout/rollback success를 주장하지 않고 `target_pending`과 owner next action을 유지합니다.